# Prophet Validation — LabOps Agent

Validates Prophet forecast accuracy on synthetic TSH demand data calibrated with real Argentine lab patterns.

**Output:** `prophet_metrics.json`

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

# Add backend to path
sys.path.insert(0, os.path.join('..', 'backend'))
import prediction

Importing plotly failed. Interactive plots will not work.


## 1. Load synthetic demand history (TSH)

Uses the same `_build_synthetic_history` function as the production backend.

In [2]:
REAGENT = 'TSH'
HISTORY_DAYS = 365

df = prediction._build_synthetic_history(REAGENT, days=HISTORY_DAYS)
print(f'Shape: {df.shape}')
print(df.head())
print(f'\nDate range: {df.ds.min()} to {df.ds.max()}')

Shape: (365, 2)
          ds      y
0 2025-06-28  134.0
1 2025-06-29  128.0
2 2025-06-30  222.0
3 2025-07-01  230.0
4 2025-07-02  213.0

Date range: 2025-06-28 00:00:00 to 2026-06-27 00:00:00


## 2. Train Prophet model

In [3]:
m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.8,
)
m.fit(df)
print('Model trained.')

02:30:21 - cmdstanpy - INFO - Chain [1] start processing


02:30:22 - cmdstanpy - INFO - Chain [1] done processing


Model trained.


## 3. Cross-validation

In [4]:
df_cv = cross_validation(
    m,
    initial='180 days',
    period='30 days',
    horizon='14 days',
    parallel='processes'
)
print(f'CV rows: {len(df_cv)}')
print(df_cv.head())

Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.


CV rows: 84
          ds        yhat  yhat_lower  yhat_upper      y     cutoff
0 2026-01-15  114.696631   95.940141  133.358588  125.0 2026-01-14
1 2026-01-16  113.171452   95.153997  132.237306  130.0 2026-01-14
2 2026-01-17   47.607481   29.664573   65.066193   82.0 2026-01-14
3 2026-01-18   51.043368   32.101445   70.097561   58.0 2026-01-14
4 2026-01-19  104.723248   87.246549  122.417974  110.0 2026-01-14


## 4. Performance metrics

In [5]:
df_p = performance_metrics(df_cv)
print(df_p[['horizon', 'mae', 'rmse', 'mape']].head())

  horizon        mae       rmse      mape
0  2 days  14.664050  17.889507  0.151004
1  3 days  14.413152  18.610437  0.152303
2  4 days  10.640349  16.618369  0.100596
3  5 days  11.952972  15.277734  0.105464
4  6 days  11.753518  16.171192  0.110076


## 5. Aggregate metrics

In [6]:
metrics = {
    'reagent': REAGENT,
    'history_days': HISTORY_DAYS,
    'cv_initial': '180 days',
    'cv_period': '30 days',
    'cv_horizon': '14 days',
    'mae': round(float(df_p['mae'].mean()), 2),
    'rmse': round(float(df_p['rmse'].mean()), 2),
    'mape': round(float(df_p['mape'].mean()), 4),
    'mape_percent': round(float(df_p['mape'].mean()) * 100, 2),
    'coverage': round(float(df_p['coverage'].mean()), 4) if 'coverage' in df_p.columns else None,
    'calibrated': True,
    'note': 'DEMO — synthetic data calibrated with real demand patterns',
}

print('=== Prophet Performance Metrics ===')
print(f'MAE  : {metrics["mae"]} units')
print(f'RMSE : {metrics["rmse"]} units')
print(f'MAPE : {metrics["mape_percent"]}%')
print(f'Coverage : {metrics["coverage"]}')

# Save to JSON
out_path = 'prophet_metrics.json'
with open(out_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nSaved to {out_path}')

=== Prophet Performance Metrics ===
MAE  : 16.51 units
RMSE : 20.6 units
MAPE : 15.75%
Coverage : 0.609



Saved to prophet_metrics.json
